# 17 — C backward (MLA + MoE)

**Before:** notebooks **15–16**.

**This notebook:** mirror `train_gpt2.c` backward in `c/deepseek_v2/` — terminal commands.

**Online course:** run cells **top-to-bottom**. Setup cell must print `data OK`.



**Before:** notebooks 15–16 (PyTorch train + sample; C forward in phase 4).

**Goal:** Mirror `train_gpt2.c` backward — piece by piece in C.

## What exists now

| Piece | Forward | Backward |
|-------|---------|----------|
| RMSNorm | `rmsnorm.c` | `ds4_rmsnorm_backward` |
| MLA | `mla.c` | `dsv2_mla_backward` |
| MoE | `moe.c` + `moe_train.c` | `dsv2_moe_backward` |
| Block | `block_train.c` | `dsv2_block_backward` (both residuals) |
| 1-layer train | `-train-1layer` | MLA + wte (MoE frozen) |
| Full train | `-train-full` | 2 layers, MLA + MoE, B=1, grad clip |

## Try it

```bash
cd c
make test_v2
./bin/train_v2_tiny -train-1layer 30
./bin/train_v2_tiny -train-full 20
```

For production-quality training use notebook 15 `Trainer`, then `scripts/export_v2_tiny.py` and `-sample -ckpt`.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import c_dir, checkpoint_path, data_path, v2_checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
C_DIR = c_dir(ROOT)
V2_CKPT = v2_checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
# PyTorch reference: MLA backward via autograd (same math C implements)
import torch
from llmc.deepseek_v2 import DeepSeekV2Config, MultiHeadLatentAttention

cfg = DeepSeekV2Config.tiny(64, 16)
attn = MultiHeadLatentAttention(cfg)
x = torch.randn(1, 8, cfg.n_embd, requires_grad=True)
y = attn(x)
y.sum().backward()
print("x.grad norm:", x.grad.norm().item())
print("wq.grad norm:", attn.wq.weight.grad.norm().item())


## Read order (C)

1. `adamw.c` — AdamW (Phase 6)
2. `ops.c` — linear + softmax backward
3. `mla.c` / `moe_train.c` / `block_train.c`
4. `model.c` — `dsv2_model_train_step_full` + `dsv2_model_save_checkpoint`

## Phase 6 commands

```bash
cd c
./bin/train_v2_tiny -train-adam 40 -batch 4 -lr 0.003 -save ../checkpoints/v2_c.bin
./bin/train_v2_tiny -sample -ckpt ../checkpoints/v2_c.bin
```
